# 05 · Coverage simulation: Fisher vs cluster bootstrap vs mixed-effects intervals

First C2 simulation. Truth is the cell-level mixed model fitted to the Stage-1 FF calendar data (per-SoC-level effects, $\tau \approx 0.2$, $\sigma \approx 0.0014$ from `04_calendar_fit.ipynb`). Each replicate redraws cell effects $b_c \sim N(0,\tau^2)$ and noise on the observed design (33 cells, 11 conditions, same check-up times), then refits and builds 95% intervals from:

- **Fisher**: Gauss–Newton covariance from least squares with a pooled residual variance;
- **cluster bootstrap**: resampling cells within condition ($B = 500$), Wald and percentile;
- **mixed Wald**: inverse numerical Hessian of the marginal likelihood.

**Scenario:** well-specified (the fitted model is the truth). Misspecified and profile-likelihood scenarios are not yet run.

Generate the draws from `src/` (about 5 minutes on 14 cores; writes `data/processed/sim_coverage_soc.parquet`):

```
cd src && python simulate_coverage.py 100 500   # replicates, bootstrap draws
```

In [ ]:
# Run the coverage simulation (overwrites ../data/processed/sim_coverage_soc.parquet)
# ~5 min per 100 replicates on 14 cores (500 replicates ~ 25 min); rerun the cells below afterwards.
import subprocess, sys

REPS = 500   # simulation replicates
B = 500      # bootstrap draws per replicate

# Run from src/ (the script uses paths relative to it) and stream its output as it runs.
# The parquet is written only when all replicates finish, so interrupting loses everything.
proc = subprocess.Popen([sys.executable, "-W", "ignore", "simulate_coverage.py", str(REPS), str(B)],
                        cwd="../src", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(f"simulation exited with code {proc.returncode}")

In [1]:
# Load simulation output; flag replicates where the mixed-model fit diverged
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from scipy import stats

s = pd.read_parquet("../data/processed/sim_coverage_soc.parquet")
r = s.groupby("rep")[["tau_hat", "sigma_hat"]].first()
# Diverged ML fit: tau -> infinity and sigma -> 0, or estimates outside a plausible range
bad = r[~np.isfinite(r["tau_hat"]) | (r["tau_hat"] > 1) | (r["sigma_hat"] < 5e-4) | (r["sigma_hat"] > 5e-3)].index
print(f"replicates: {r.shape[0]} | diverged mixed fits excluded: {len(bad)} {list(bad)}")
g0 = s[~s["rep"].isin(bad)]
print(g0.groupby("rep")[["tau_hat", "sigma_hat"]].first().describe().loc[["mean", "std", "min", "max"]].round(4))

replicates: 100 | diverged mixed fits excluded: 5 [5, 34, 37, 46, 91]
      tau_hat  sigma_hat
mean   0.1824     0.0014
std    0.0315     0.0001
min    0.0870     0.0011
max    0.2610     0.0017


In [2]:
# Empirical SE vs mean estimated SE, and 95% coverage, per parameter (excluding diverged replicates)
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from scipy import stats

s = pd.read_parquet("../data/processed/sim_coverage_soc.parquet")
r = s.groupby("rep")[["tau_hat", "sigma_hat"]].first()
bad = r[~np.isfinite(r["tau_hat"]) | (r["tau_hat"] > 1) | (r["sigma_hat"] < 5e-4) | (r["sigma_hat"] > 5e-3)].index
g0 = s[~s["rep"].isin(bad)]

def cover(lo, hi, t):
    return ((lo <= t) & (t <= hi)).mean()

rows = []
for p, g in g0.groupby("param", sort=False):
    t = g["truth"].iloc[0]
    rows.append({"param": p, "n": len(g),
                 "SD(est) NLS": g["est_nls"].std(), "SD(est) mixed": g["est_mixed"].std(),
                 "Fisher SE": g["se_fisher"].mean(), "boot SE": g["se_boot"].mean(), "mixed SE": g["se_mixed"].mean(),
                 "cov Fisher": cover(g["est_nls"] - 1.96 * g["se_fisher"], g["est_nls"] + 1.96 * g["se_fisher"], t),
                 "cov boot Wald": cover(g["est_nls"] - 1.96 * g["se_boot"], g["est_nls"] + 1.96 * g["se_boot"], t),
                 "cov boot pct": cover(g["pct_lo"], g["pct_hi"], t),
                 "cov mixed": cover(g["est_mixed"] - 1.96 * g["se_mixed"], g["est_mixed"] + 1.96 * g["se_mixed"], t)})
tab = pd.DataFrame(rows).set_index("param")
tab.round(3)

,n,SD(est) NLS,SD(est) mixed,Fisher SE,boot SE,mixed SE,cov Fisher,cov boot Wald,cov boot pct,cov mixed
param,,,,,,,,,,
log_k_ref,95,0.097,0.082,0.052,0.072,0.066,0.674,0.800,0.789,0.863
Ea_kJ,95,3.037,2.497,1.116,2.218,2.025,0.547,0.832,0.821,0.905
delta_soc0.05,95,0.126,0.109,0.121,0.103,0.105,0.916,0.800,0.811,0.947
delta_soc0.20,95,0.167,0.161,0.091,0.130,0.137,0.705,0.821,0.821,0.916
delta_soc0.80,95,0.166,0.161,0.049,0.120,0.133,0.442,0.832,0.800,0.853
delta_soc1.00,95,0.119,0.099,0.046,0.095,0.089,0.484,0.895,0.884,0.905
z,95,0.046,0.034,0.080,0.036,0.034,0.989,0.811,0.811,0.947


**Findings** (from the 100-replicate run: 95 usable replicates; coverage Monte Carlo SE is about 0.02–0.03).
- Fisher intervals badly undercover the rate parameters (about 0.45–0.55 for $E_a$ and the SoC 0.8 and 1.0 effects) and overcover $z$ (0.99): the pooled residual variance ignores cell clustering.
- The cluster bootstrap undercovers uniformly at about 0.80–0.88, consistent with only 3 cells per condition.
- Mixed Wald intervals are closest to nominal (0.85–0.95) when the model is correct.
- Estimated τ averages about 0.18 against a truth of 0.206 (ML is biased low with 33 cells).

**Limits.** The mixed-model optimizer diverged in 5 of 100 replicates (no bounds on log σ, log τ); these are excluded, not fixed. Only the well-specified scenario has been run. Next: bounds and restarts, 500 replicates, a misspecified scenario (simulate from the saturated mixed model, fit the per-SoC model), and profile-likelihood intervals.